In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
sqlite_index.count()

85

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

**RAG with sqlitesearch**

We use the `RAGBase` class from `rag_helper.py` with this sqlitesearch index.

Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [4]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, make sure to submit your project while submissions are still open.


**Choosing an approach**

Pick the right tool for your data:

- minsearch: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
- sqlitesearch: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different backend:

- Elasticsearch
- OpenSearch
- Qdrant (vector database)
- Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

**Cleaning up**

When you're done, close the database connection:

In [6]:
sqlite_index.close()